# Sistema Recomendador por Clientes Similares
## User-Based Collaborative Filtering (UBCF)

**Caso**: Recomendar productos a clientes B2B a partir de clientes con comportamiento similar
**Algoritmo**: K-Nearest Neighbors sobre matriz cliente x producto con similitud coseno
**Target**: score_recomendacion(cliente_u, producto_p) - ranking topN_productos por cliente

> El target no existe en el dataset: lo genera el modelo a partir del historial.
> La interpretacion del ranking es trabajo del equipo de negocio.

**Lo que aprenderas:**
1. Como construir una matriz cliente-producto desde logs de interacciones
2. Por que la similitud coseno funciona mejor que Pearson aqui
3. Como implementar predict() en un sistema sin etiquetas
4. Como evaluar un recomendador (Precision@K, cold-start)

---

## Contexto del caso de negocio

| | |
|---|---|
| **Empresa** | empresa — área de producto y expansión de cuenta |
| **Problema de negocio** | Recomendar a cada cliente los productos que probablemente le interesen, basándose en el comportamiento de clientes con un perfil de uso similar, sin necesidad de que el cliente haya interactuado previamente con esos productos |
| **Datos disponibles** | Catálogo de 40 productos y 4.800 interacciones cliente-producto (valoraciones implícitas de uso) que forman una matriz dispersa de preferencias |
| **Técnica aplicada** | Filtrado colaborativo basado en usuario (UBCF) con KNN y similitud coseno: para un cliente dado, encuentra los K vecinos más similares y agrega las interacciones de esos vecinos para producir recomendaciones |
| **Salida del modelo** | Lista de productos recomendados para un cliente concreto, ordenados por relevancia estimada a partir del comportamiento de clientes similares |
| **Valor operativo** | Permite al equipo de ventas y customer success presentar sugerencias de productos personalizadas en cada interacción, aumentando la tasa de adopción de nuevos servicios sin necesidad de segmentación manual |

In [ ]:
import os, sys
from pathlib import Path

# Configuracion de entorno: ajusta CWD y descarga datos segun el entorno de ejecucion
_BASE_URL = "https://raw.githubusercontent.com/amador2001/ia-datos/main/"
_CSVS = ["catalogo_productos.csv", "interacciones_cliente_producto.csv"]

if "google.colab" in sys.modules:
    import urllib.request
    Path("datos").mkdir(exist_ok=True)
    for _csv in _CSVS:
        _dest = Path("datos") / _csv
        if not _dest.exists():
            urllib.request.urlretrieve(_BASE_URL + _csv, str(_dest))
            print(f"Descargado: {_csv}")
elif "__vsc_ipynb_file__" in dir():
    os.chdir(Path(__vsc_ipynb_file__).parent)
elif not Path("datos").exists():
    for _p in [Path("Jupyter_notebooks"), Path("../Jupyter_notebooks")]:
        if (_p / "datos").exists():
            os.chdir(_p)
            break

print(f"Entorno listo. CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 110

In [ ]:
# Cargar el catalogo de productos
catalogo = pd.read_csv("datos/catalogo_productos.csv")
print("Catalogo de productos:")
display(catalogo.head(10))

# Cargar el historico de interacciones cliente-producto
interacciones = pd.read_csv("datos/interacciones_cliente_producto.csv")
print(f"\nInteracciones (primeras filas):")
display(interacciones.head(10))

print(f"\nResumen del dataset:")
print(f"  Productos en catalogo: {len(catalogo)}")
print(f"  Clientes unicos:       {interacciones['cliente_id'].nunique()}")
print(f"  Interacciones totales: {len(interacciones)}")
print(f"  Tipos de evento: {interacciones['evento'].value_counts().to_dict()}")
print(f"  Pesos por evento: view=1, add_to_cart=3, buy=5")

### Como funciona el Collaborative Filtering

El sistema parte de una **matriz cliente x producto** donde cada celda contiene
el score de interaccion (suma de pesos de todos los eventos del cliente con ese producto).

```
           PROD-001  PROD-002  PROD-003  ...
USR-0001      0         5         1
USR-0002      3         0         8
USR-0003      0         2         0
```

La mayoria de celdas son 0 (sparsity ~66%). El algoritmo:
1. Normaliza cada fila (vector de cliente) para que la escala no distorsione la similitud
2. Calcula la similitud coseno entre todos los pares de clientes
3. Para un cliente objetivo, encuentra los K vecinos mas similares
4. Recomienda los productos que esos vecinos consumen y el objetivo no ha visto aun

**Por que similitud coseno y no correlacion de Pearson?**
Con datos implicitos (vistas, clics), la mayoria de valores son 0.
El coseno es mas robusto ante celdas vacias que Pearson.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Distribucion de interacciones por cliente
inter_por_cliente = interacciones.groupby("cliente_id").size()
inter_por_cliente.plot.hist(bins=20, ax=axes[0,0], color="steelblue", edgecolor="white")
axes[0,0].set_title("Interacciones por cliente")
axes[0,0].set_xlabel("Numero de interacciones")

# Power law de productos: pocos productos concentran la mayoria de interacciones
inter_por_prod = interacciones.groupby("producto_id").size().sort_values(ascending=False)
axes[0,1].bar(range(len(inter_por_prod)), inter_por_prod.values, color="steelblue")
axes[0,1].set_title("Distribucion de interacciones por producto (power law)")
axes[0,1].set_xlabel("Producto (ordenado por popularidad)")
axes[0,1].set_ylabel("Interacciones")

# Distribucion de tipos de evento
interacciones["evento"].value_counts().plot(kind="pie", ax=axes[1,0],
    colors=["steelblue","orange","green"], autopct="%1.0f%%")
axes[1,0].set_title("Distribucion de eventos")

# Interacciones por mes
interacciones["mes"] = pd.to_datetime(interacciones["timestamp"]).dt.to_period("M").astype(str)
interacciones.groupby("mes").size().plot(ax=axes[1,1], color="steelblue")
axes[1,1].set_title("Volumen de interacciones por mes")
axes[1,1].set_xlabel("Mes")
axes[1,1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# -- Construir matriz cliente x producto (pivot) -----------------------------------------------
# Agregar por (cliente, producto): suma de pesos de todos los eventos
matriz_raw = (interacciones
    .groupby(["cliente_id","producto_id"])["peso_evento"]
    .sum()
    .unstack(fill_value=0))   # fill_value=0: si no hay interaccion, el score es 0

print(f"Dimensiones de la matriz cliente-producto: {matriz_raw.shape}")
print(f"Sparsity: {(matriz_raw==0).sum().sum() / matriz_raw.size:.1%} de celdas son 0")
print("\nPrimeras filas de la matriz (primeros 8 productos):")
display(matriz_raw.iloc[:5, :8])

# -- Split temporal: holdout del 20% de interacciones para evaluacion --------------------------
# Ordenar por timestamp y usar las ultimas interacciones como test
interacciones_sorted = interacciones.sort_values("timestamp")
n_test = int(len(interacciones_sorted) * 0.20)
train_inter = interacciones_sorted.iloc[:-n_test]
test_inter  = interacciones_sorted.iloc[-n_test:]

print(f"\nInteracciones de entrenamiento: {len(train_inter)}")
print(f"Interacciones de test:          {len(test_inter)}")

# Reconstruir la matriz solo con interacciones de entrenamiento
matriz_train = (train_inter
    .groupby(["cliente_id","producto_id"])["peso_evento"]
    .sum()
    .unstack(fill_value=0))
matriz_train = matriz_train.reindex(index=matriz_raw.index,
                                     columns=matriz_raw.columns, fill_value=0)

# -- Normalizar: cada cliente = vector unitario ------------------------------------------------
# normalize() divide cada fila por su norma L2
# Asi la similitud coseno no se ve afectada por la actividad total del cliente
# (un cliente muy activo no domina sobre uno poco activo)
matriz_norm = normalize(matriz_train.values, norm="l2")

print("\nDataset normalizado (primeras 3 filas, primeras 6 columnas):")
display(pd.DataFrame(matriz_norm[:3, :6],
        index=matriz_train.index[:3],
        columns=matriz_train.columns[:6]).round(4))

In [ ]:
# -- Modelo: KNN con similitud coseno ----------------------------------------------------------
# metric="cosine" calcula 1 - similitud_coseno como distancia
# n_neighbors=10: los 10 clientes mas similares para hacer recomendaciones
K_VECINOS = 10
knn = NearestNeighbors(n_neighbors=K_VECINOS + 1,  # +1 porque incluye el propio cliente
                        metric="cosine",
                        algorithm="brute")
knn.fit(matriz_norm)   # aprende la estructura de distancias del espacio de clientes

def recomendar(cliente_id, top_n=5):
    """
    Devuelve los top_n productos recomendados para un cliente dado.
    
    Algoritmo:
    1. Buscar los K clientes mas similares
    2. Sumar sus scores ponderados por similitud
    3. Filtrar los productos que el cliente ya conoce
    4. Devolver el ranking de los mejores
    """
    if cliente_id not in matriz_train.index:
        return pd.DataFrame(columns=["producto_id","score","nombre","categoria"])
    
    idx = list(matriz_train.index).index(cliente_id)
    vec = matriz_norm[idx].reshape(1, -1)
    
    # kneighbors devuelve (distancias, indices) de los K vecinos mas cercanos
    distancias, indices = knn.kneighbors(vec)
    
    # Convertir distancia coseno a similitud: similitud = 1 - distancia
    similitudes = 1 - distancias[0][1:]   # excluir el propio cliente (indice 0)
    vecinos_idx = indices[0][1:]
    
    # Productos ya vistos por el cliente (no recomendar lo que ya conoce)
    ya_conocidos = set(matriz_train.columns[matriz_train.iloc[idx] > 0])
    
    # Calcular score de recomendacion: promedio ponderado por similitud
    scores = np.zeros(len(matriz_train.columns))
    for sim, v_idx in zip(similitudes, vecinos_idx):
        scores += sim * matriz_norm[v_idx]
    
    # Crear DataFrame con scores y filtrar ya conocidos
    rec = pd.DataFrame({
        "producto_id": matriz_train.columns,
        "score": scores
    })
    rec = rec[~rec["producto_id"].isin(ya_conocidos)]
    rec = rec.sort_values("score", ascending=False).head(top_n)
    
    # Enriquecer con nombre y categoria del catalogo
    rec = rec.merge(catalogo[["producto_id","nombre","categoria","precio_mensual"]],
                    on="producto_id", how="left")
    return rec.reset_index(drop=True)

# -- Prediccion: recomendaciones para 3 clientes de ejemplo ------------------------------------
for cli in ["USR-0001", "USR-0025", "USR-0099"]:
    print(f"\nTop-5 recomendaciones para {cli}:")
    display(recomendar(cli, top_n=5))

In [ ]:
# Heatmap de similitud coseno entre primeros 20 clientes
sim_matrix = cosine_similarity(matriz_norm[:20])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Similitud coseno entre primeros 20 clientes")
axes[0].set_xlabel("Cliente"); axes[0].set_ylabel("Cliente")
plt.colorbar(im, ax=axes[0])

# Distribucion de similitudes (excluyendo diagonal)
sims_off = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
axes[1].hist(sims_off, bins=30, color="steelblue", edgecolor="white")
axes[1].set_title("Distribucion de similitudes coseno")
axes[1].set_xlabel("Similitud coseno")
axes[1].set_ylabel("Frecuencia")
axes[1].axvline(sims_off.mean(), color="red", linestyle="--",
                label=f"Media: {sims_off.mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

### Limitaciones del recomendador colaborativo

| Problema | Descripcion | Solucion parcial |
|---|---|---|
| Cold-start | Cliente nuevo sin historial: no hay vecinos | Recomendar por popularidad global |
| Popularity bias | Tiende a recomendar lo popular | Penalizar items muy frecuentes |
| Sparsity | Con 66% de ceros, las similitudes son bajas | Aumentar el historico de interacciones |
| Escalabilidad | Con 10k+ clientes, calcular similitudes es lento | Approximate Nearest Neighbors (FAISS) |

> **Cuando este modelo NO tiene valor**:
> si todos los clientes compran los mismos productos (baja diversidad),
> la similitud entre clientes no aporta informacion y es mejor usar
> popularidad global o reglas de negocio.